# Experiment 1 Colab Runner

This notebook is built for interactive phase-by-phase work.

Why this notebook is structured this way:
- one cell loads or updates the repo
- one cell defines the experiment config directly in Colab
- one cell previews synthetic training targets before training
- one cell runs a single phase
- one cell inspects saved metrics/examples for that phase
- one optional cell runs the full experiment loop

This makes it easier to debug one phase at a time instead of running the whole pipeline blindly.

In [ ]:
# Set RUN_ENV to 'local' on your computer or 'colab' in Google Colab.
RUN_ENV = 'colab'

GIT_REPO_URL = 'https://github.com/ffbskt/AgentAI.git'
LOCAL_REPO_DIR = r'D:/Codex projects/Transformer_Graph3/arithmetic-transformer'
COLAB_PROJECT_ROOT = '/content/drive/MyDrive/AgentAI'

In [ ]:
if RUN_ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if RUN_ENV == 'colab':
    project_root = Path(COLAB_PROJECT_ROOT)
    repo_root = project_root
    git_dir = repo_root / '.git'

    if git_dir.exists():
        print(f"'{repo_root}' is already a git repository. Pulling latest changes...")
        subprocess.run(['git', '-C', str(repo_root), 'pull'], check=True)
    elif repo_root.exists():
        print(f"'{repo_root}' exists but is not a git repository. Removing and re-cloning...")
        shutil.rmtree(repo_root)
        project_root.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', GIT_REPO_URL, str(repo_root)], check=True)
    else:
        print(f"Cloning '{GIT_REPO_URL}' into '{repo_root}'...")
        project_root.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', GIT_REPO_URL, str(repo_root)], check=True)

    REPO_DIR = repo_root / 'arithmetic-transformer'
else:
    REPO_DIR = Path(LOCAL_REPO_DIR)

EXPERIMENT_DIR = REPO_DIR / 'experiment_1'
LOG_ROOT = EXPERIMENT_DIR / 'logs' / 'colab_run'
LOG_ROOT.mkdir(parents=True, exist_ok=True)

print('Repo dir:', REPO_DIR)
print('Experiment dir:', EXPERIMENT_DIR)
print('Log root:', LOG_ROOT)

In [ ]:
%cd {REPO_DIR}
!python -m pip install -q -r requirements-colab.txt

## Editable experiment config

Why keep config in a notebook cell:
- fast changes in Colab without editing repo files
- easy to try a different phase set or training budget
- the Python runner accepts this config as inline JSON

In [ ]:
import json
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PHASE_OUTPUT_ROOT = 'logs/colab_run'

EXPERIMENT_CONFIG = {
    'model': {
        'kind': 'transformer-sine',
        'hidden_size': 64,
        'ffw_size': 128,
        'num_layers': 2,
        'num_heads': 4,
        'dropout': 0.0,
        'lr': 0.001
    },
    'train_defaults': {
        'epochs': 1,
        'batch_size': 64,
        'train_samples': 256,
        'val_samples': 128,
        'test_samples': 128,
        'max_new_tokens': 48,
        'rl_steps': 4,
        'rl_batch_size': 16,
        'num_generations': 4,
        'temperature': 1.0,
        'kl_coef': 0.02,
        'best_of_n_steps': 2,
        'seed': 11
    },
    'phases': [
        {'name': 'step1_1d', 'description': '1-digit warmup', 'fmt': 'A', 'shape': '1d+1d', 'carry_mode': 'any', 'rl_enabled': True},
        {'name': 'step2_2d', 'description': '2-digit reasoning', 'fmt': 'C', 'shape': '2d+2d', 'carry_mode': 'simple', 'rl_enabled': True},
        {'name': 'step3_3d', 'description': '3-digit reasoning', 'fmt': 'C', 'shape': '3d+3d', 'carry_mode': 'any', 'rl_enabled': True},
        {'name': 'step4_4d', 'description': '4-digit reasoning', 'fmt': 'C', 'shape': '4d+4d', 'carry_mode': 'any', 'rl_enabled': True},
        {'name': 'step5_5d', 'description': '5-digit reasoning', 'fmt': 'C', 'shape': '5d+5d', 'carry_mode': 'heavy', 'rl_enabled': True}
    ]
}

CONFIG_JSON = json.dumps(EXPERIMENT_CONFIG)
print('Selected device:', DEVICE)
print('Configured phases:', [phase['name'] for phase in EXPERIMENT_CONFIG['phases']])

## Import the phase-oriented runner functions

Why these functions exist:
- `load_experiment_config`: build usable phase objects from notebook config
- `preview_phase_samples`: inspect expected symbolic targets before training
- `run_one_phase`: train and evaluate one phase only
- `run_experiment`: loop over all phases when you want the full pipeline

In [ ]:
%cd {EXPERIMENT_DIR}
from pathlib import Path
from train_experiment_1 import (
    load_experiment_config,
    preview_phase_samples,
    run_one_phase,
    run_experiment,
)

raw_config, phases = load_experiment_config(config_json=CONFIG_JSON)
print('Loaded phases:', [phase.name for phase in phases])

## Preview one phase before training

This is useful because RLVR depends on deterministic target style.
If the generated symbolic targets already look wrong here, training will also be wrong.

In [ ]:
PHASE_INDEX = 0  # Change this to inspect a different phase.
preview = preview_phase_samples(phases[PHASE_INDEX], sample_count=5)
for item in preview:
    print(item)


## Run one phase only

This cell is the most important one for debugging.
Use it when you want to train exactly one phase, save its outputs, and inspect them before continuing.

In [ ]:
import torch

PHASE_INDEX = 0  # Change this to run another phase.
phase = phases[PHASE_INDEX]
phase_output_dir = Path(PHASE_OUTPUT_ROOT) / phase.name

model = None
model, summary = run_one_phase(
    phase=phase,
    model=model,
    device=torch.device(DEVICE),
    output_root=Path(PHASE_OUTPUT_ROOT),
    sample_limit=32,
)
summary

## Inspect saved phase artifacts

Keep training and inspection separate.
This helps answer questions like:
- did SFT improve anything?
- did RLVR help or hurt?
- what examples is the model producing?

In [ ]:
import json

phase_dir = Path(PHASE_OUTPUT_ROOT) / phases[PHASE_INDEX].name
print('Phase dir:', phase_dir)
print('\nSummary:')
print(json.loads((phase_dir / 'summary.json').read_text(encoding='utf-8')))

print('\nPost-SFT examples:')
print(json.loads((phase_dir / 'post_sft_examples.json').read_text(encoding='utf-8'))[:2])

print('\nPost-RLVR examples:')
print(json.loads((phase_dir / 'post_rlvr_examples.json').read_text(encoding='utf-8'))[:2])

## Run the full experiment loop

Use this only after single-phase checks look sane.
It runs all configured phases in order and writes a top-level summary.

In [ ]:
summaries = run_experiment(
    config_json=CONFIG_JSON,
    device=DEVICE,
    output_dir=PHASE_OUTPUT_ROOT,
)
summaries

In [ ]:
!ls -lah {PHASE_OUTPUT_ROOT}
!cat {PHASE_OUTPUT_ROOT}/summary.csv || true